# HiLo Breakout EA — Trade Analysis

Reads `HiLo_trades_<MagicNumber>.csv` files written by the EA and produces:
- Per-magic overview stats
- P&L and win rate by hour of day
- Slippage analysis by hour (ATR Candle method)
- Session performance (London, NY, Tokyo, Sydney)
- Filter rejection analysis
- SL size distribution and MaxSLPips threshold sensitivity
- Concurrent order detection across instances

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('Libraries loaded OK')

## Configuration

**Set `MT4_FILES_PATH` to the MT4 Files folder on your PC.**

To find the path in MT4: go to **File → Open Data Folder**, then navigate into `MQL4 → Files`.

Typical path: `C:\Users\YourName\AppData\Roaming\MetaQuotes\Terminal\<ID>\MQL4\Files`

In [ ]:
# ── SET THIS PATH ─────────────────────────────────────────────────────────────
MT4_FILES_PATH = r"C:\Users\YourName\AppData\Roaming\MetaQuotes\Terminal\XXXXXXXX\MQL4\Files"

# Optional: restrict to specific magic numbers (leave as [] to load all)
MAGIC_FILTER = []  # e.g. [123456, 234567]

# Seconds threshold for flagging concurrent orders across instances
CONCURRENT_SECONDS = 5
# ──────────────────────────────────────────────────────────────────────────────

## Load Data

In [ ]:
files = list(Path(MT4_FILES_PATH).glob('HiLo_trades_*.csv'))

if not files:
    print(f'ERROR: No trade log files found in:\n  {MT4_FILES_PATH}')
    print('\nCheck MT4_FILES_PATH in the configuration cell above.')
else:
    dfs = []
    for f in sorted(files):
        magic = int(f.stem.split('_')[-1])
        if MAGIC_FILTER and magic not in MAGIC_FILTER:
            continue
        df = pd.read_csv(f)
        for col in ['log_time', 'signal_time', 'close_time']:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
        df['magic'] = magic
        dfs.append(df)
        print(f'Loaded {len(df):>5} rows  from  {f.name}')

    data = pd.concat(dfs, ignore_index=True)

    # Split by event type
    entries   = data[data['event'] == 'ENTRY'].copy()
    closes    = data[data['event'] == 'CLOSE'].copy()
    rejected  = data[data['event'] == 'REJECTED'].copy()
    cancelled = data[data['event'] == 'CANCELLED'].copy()

    # Full trade view: join ENTRY with CLOSE on ticket
    close_cols = ['ticket', 'close_price', 'close_time', 'close_reason',
                  'profit_pips', 'profit_usd']
    trades = entries.merge(
        closes[close_cols].rename(columns={
            'profit_pips': 'profit_pips',
            'profit_usd':  'profit_usd'
        }),
        on='ticket', how='left', suffixes=('', '_close')
    )
    # Prefer close-row values for profit fields (they may also appear in entry cols)
    for col in ['close_price', 'close_time', 'close_reason', 'profit_pips', 'profit_usd']:
        if f'{col}_close' in trades.columns:
            trades[col] = trades[f'{col}_close'].combine_first(trades.get(col))
            trades.drop(columns=[f'{col}_close'], inplace=True)

    trades['closed'] = trades['close_price'].notna()
    trades['win']    = trades['profit_pips'] > 0

    # Flag concurrent orders: entries placed within CONCURRENT_SECONDS across magics
    if len(entries) > 1:
        all_entries = trades[['ticket', 'magic', 'log_time']].sort_values('log_time').reset_index(drop=True)
        conc = set()
        for i in range(len(all_entries)):
            for j in range(i + 1, len(all_entries)):
                row_i, row_j = all_entries.iloc[i], all_entries.iloc[j]
                if (row_j['log_time'] - row_i['log_time']).total_seconds() > CONCURRENT_SECONDS:
                    break
                if row_i['magic'] != row_j['magic']:
                    conc.add(row_i['ticket'])
                    conc.add(row_j['ticket'])
        trades['concurrent'] = trades['ticket'].isin(conc)
    else:
        trades['concurrent'] = False

    closed_trades = trades[trades['closed']]
    print(f'\nTotal entries: {len(trades)} | Closed: {len(closed_trades)} '
          f'| Wins: {trades["win"].sum()} | Losses: {(~trades["win"] & trades["closed"]).sum()}')
    print(f'Rejections: {len(rejected)} | Cancellations: {len(cancelled)} '
          f'| Concurrent flags: {trades["concurrent"].sum()}')

## Overview by Magic Number

In [ ]:
def win_rate(group):
    closed = group[group['closed']]
    return round(closed['win'].mean() * 100, 1) if len(closed) > 0 else float('nan')

def avg_win(group):
    w = group.loc[group['win'] & group['closed'], 'profit_pips']
    return w.mean() if len(w) > 0 else float('nan')

def avg_loss(group):
    l = group.loc[~group['win'] & group['closed'], 'profit_pips']
    return l.mean() if len(l) > 0 else float('nan')

summary = trades.groupby(['magic', 'method']).apply(lambda g: pd.Series({
    'entries':        len(g),
    'closed':         g['closed'].sum(),
    'open':           (~g['closed']).sum(),
    'win_rate_%':     win_rate(g),
    'total_pips':     g['profit_pips'].sum(),
    'total_usd':      g['profit_usd'].sum(),
    'avg_win_pips':   avg_win(g),
    'avg_loss_pips':  avg_loss(g),
    'concurrent':     g['concurrent'].sum(),
})).reset_index()

display(summary.style
    .format({
        'win_rate_%':    '{:.1f}%',
        'total_pips':    '{:.1f}',
        'total_usd':     '${:.2f}',
        'avg_win_pips':  '{:.1f}',
        'avg_loss_pips': '{:.1f}',
    }, na_rep='-')
    .set_caption('Trade Summary by Magic Number and Method')
    .background_gradient(subset=['win_rate_%', 'total_usd'], cmap='RdYlGn')
)

## Equity Curve by Magic Number

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for magic, group in closed_trades.groupby('magic'):
    g = group.sort_values('close_time')
    axes[0].plot(g['close_time'], g['profit_pips'].cumsum(), label=f'Magic {magic}', marker='.', markersize=3)
    axes[1].plot(g['close_time'], g['profit_usd'].cumsum(),  label=f'Magic {magic}', marker='.', markersize=3)

axes[0].set_title('Cumulative P&L (pips)')
axes[0].set_ylabel('Pips')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].legend()

axes[1].set_title('Cumulative P&L (USD)')
axes[1].set_ylabel('USD')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].legend()

for ax in axes:
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## P&L and Win Rate by Hour of Day

Hours are server time as recorded by the EA. Use this to assess which `ATRCandle_TW` time windows to enable.

In [ ]:
magics = trades['magic'].unique()
n = len(magics)
fig, axes = plt.subplots(n, 2, figsize=(16, 4 * n), squeeze=False)

for idx, magic in enumerate(sorted(magics)):
    ct = closed_trades[closed_trades['magic'] == magic].copy()

    hourly_pips = ct.groupby('hour')['profit_pips'].sum().reindex(range(24), fill_value=0)
    hourly_wr   = ct.groupby('hour')['win'].mean().reindex(range(24), fill_value=np.nan) * 100
    counts      = ct.groupby('hour')['win'].count().reindex(range(24), fill_value=0)

    ax0, ax1 = axes[idx]

    colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in hourly_pips]
    ax0.bar(range(24), hourly_pips, color=colors, edgecolor='white')
    ax0.set_title(f'Magic {magic} — Total P&L per Hour (pips)')
    ax0.set_xlabel('Server Hour')
    ax0.set_ylabel('Pips')
    ax0.set_xticks(range(24))
    ax0.axhline(0, color='black', linewidth=0.8)
    for x, c in zip(range(24), counts):
        if c > 0:
            ax0.text(x, 0.5, str(c), ha='center', va='bottom', fontsize=8, color='gray')

    ax1.bar(range(24), hourly_wr, color='steelblue', edgecolor='white')
    ax1.axhline(50, color='orange', linewidth=1, linestyle='--', label='50%')
    ax1.set_title(f'Magic {magic} — Win Rate per Hour (%)')
    ax1.set_xlabel('Server Hour')
    ax1.set_ylabel('Win Rate %')
    ax1.set_xticks(range(24))
    ax1.set_ylim(0, 105)
    ax1.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Slippage by Hour of Day

For the **ATR Candle** method, `slippage_pips` = entry price vs signal candle close price.  
Positive = entry was worse than signal (normal in fast markets). Large positive values at certain hours indicate high-impact news times or low-liquidity windows.

For ZZ Semafor stop orders, slippage is 0 by design.

In [ ]:
atr_trades = trades[(trades['method'] == 'ATRCandle') & trades['slippage_pips'].notna()]

if atr_trades.empty:
    print('No ATR Candle trades with slippage data found.')
else:
    magics_atr = atr_trades['magic'].unique()
    fig, axes = plt.subplots(1, len(magics_atr), figsize=(14, 5), squeeze=False)

    for idx, magic in enumerate(sorted(magics_atr)):
        g = atr_trades[atr_trades['magic'] == magic]
        slp_hour = g.groupby('hour')['slippage_pips'].agg(['mean', 'max', 'count'])

        ax = axes[0][idx]
        ax.bar(slp_hour.index, slp_hour['mean'], color='coral', label='Avg slippage', edgecolor='white')
        ax.plot(slp_hour.index, slp_hour['max'], 'k--', marker='x', linewidth=1, label='Max slippage')
        ax.set_title(f'Magic {magic} — ATR Candle Slippage by Hour')
        ax.set_xlabel('Server Hour')
        ax.set_ylabel('Pips')
        ax.set_xticks(range(24))
        ax.axhline(0, color='black', linewidth=0.8)
        ax.legend(fontsize=9)
        for x, row in slp_hour.iterrows():
            if row['count'] > 0:
                ax.text(x, 0.1, str(int(row['count'])), ha='center', va='bottom', fontsize=7, color='gray')

    plt.tight_layout()
    plt.show()

    print('\nSlippage summary (ATR Candle trades):')
    print(atr_trades.groupby('magic')['slippage_pips'].describe().round(2))

## Performance by Trading Session

In [ ]:
SESSION_ORDER = ['Tokyo', 'London_Tokyo', 'London', 'London_NY', 'New_York', 'Sydney_Tokyo', 'Sydney', 'Off']

for magic in sorted(trades['magic'].unique()):
    ct = closed_trades[closed_trades['magic'] == magic]
    if ct.empty:
        continue

    sess_stats = ct.groupby('session').agg(
        trades   = ('win', 'count'),
        win_rate = ('win', lambda x: round(x.mean() * 100, 1)),
        total_pips = ('profit_pips', 'sum'),
        avg_pips   = ('profit_pips', 'mean'),
    ).reindex([s for s in SESSION_ORDER if s in ct['session'].unique()])

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f'Magic {magic} — Session Performance', fontsize=13, fontweight='bold')

    axes[0].barh(sess_stats.index, sess_stats['trades'], color='steelblue', edgecolor='white')
    axes[0].set_title('Number of Trades')
    axes[0].set_xlabel('Count')

    colors = ['#2ecc71' if v >= 50 else '#e74c3c' for v in sess_stats['win_rate']]
    axes[1].barh(sess_stats.index, sess_stats['win_rate'], color=colors, edgecolor='white')
    axes[1].axvline(50, color='orange', linewidth=1, linestyle='--')
    axes[1].set_title('Win Rate (%)')
    axes[1].set_xlabel('%')
    axes[1].set_xlim(0, 105)

    colors2 = ['#2ecc71' if v >= 0 else '#e74c3c' for v in sess_stats['total_pips']]
    axes[2].barh(sess_stats.index, sess_stats['total_pips'], color=colors2, edgecolor='white')
    axes[2].axvline(0, color='black', linewidth=0.8)
    axes[2].set_title('Total P&L (pips)')
    axes[2].set_xlabel('Pips')

    plt.tight_layout()
    plt.show()

    display(sess_stats.style
        .format({'win_rate': '{:.1f}%', 'total_pips': '{:.1f}', 'avg_pips': '{:.1f}'})
        .background_gradient(subset=['win_rate', 'total_pips'], cmap='RdYlGn')
    )

## Filter Rejection Analysis

Shows what the filters are blocking. If `ER_LAYER1` is blocking a large proportion of signals, consider whether those would have been profitable — that data is not available here but the hours and sessions they are rejected in gives a clue.

In [ ]:
if rejected.empty:
    print('No rejection events recorded.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    rej_by_reason = rejected.groupby(['magic', 'rejection_reason']).size().unstack(fill_value=0)
    rej_by_reason.plot(kind='bar', ax=axes[0], edgecolor='white')
    axes[0].set_title('Rejection Count by Reason and Magic')
    axes[0].set_xlabel('Magic Number')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=0)
    axes[0].legend(title='Reason', fontsize=9)

    rej_by_hour = rejected.groupby('hour').size().reindex(range(24), fill_value=0)
    axes[1].bar(range(24), rej_by_hour, color='salmon', edgecolor='white')
    axes[1].set_title('Rejection Count by Hour of Day')
    axes[1].set_xlabel('Server Hour')
    axes[1].set_ylabel('Count')
    axes[1].set_xticks(range(24))

    plt.tight_layout()
    plt.show()

    total = len(rejected) + len(trades)
    print(f'\nRejection rate: {len(rejected)/total*100:.1f}% of all signals')
    print(f'\nRejections by reason:')
    print(rejected['rejection_reason'].value_counts())

if not cancelled.empty:
    print(f'\nER Layer 2 cancellations by hour:')
    print(cancelled.groupby('hour').size())

## SL Size Analysis

Distribution of SL sizes for placed trades. The vertical line shows the current `ATRCandle_MaxSLPips` threshold.  
Use the sensitivity analysis to see the P&L impact of different threshold values.

In [ ]:
if trades['sl_pips'].notna().any():
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for magic, group in trades.groupby('magic'):
        sl_data = group['sl_pips'].dropna()
        axes[0].hist(sl_data, bins=30, alpha=0.6, label=f'Magic {magic}', edgecolor='white')

    axes[0].set_title('SL Size Distribution (pips)')
    axes[0].set_xlabel('SL Pips')
    axes[0].set_ylabel('Count')
    axes[0].legend()

    # Profit vs SL size scatter (closed trades only)
    ct_sl = closed_trades[closed_trades['sl_pips'].notna()]
    for magic, group in ct_sl.groupby('magic'):
        axes[1].scatter(group['sl_pips'], group['profit_pips'],
                        alpha=0.5, s=20, label=f'Magic {magic}')

    axes[1].axhline(0, color='black', linewidth=0.8)
    axes[1].set_title('Profit (pips) vs SL Size')
    axes[1].set_xlabel('SL Pips')
    axes[1].set_ylabel('Profit Pips')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('No SL pips data in trade entries.')

### MaxSLPips Threshold Sensitivity

Simulates applying different `ATRCandle_MaxSLPips` values to the historical ATR Candle trades.  
Trades above the threshold would have been rejected — shows the cumulative P&L impact.

In [ ]:
atr_closed = closed_trades[(closed_trades['method'] == 'ATRCandle') & closed_trades['sl_pips'].notna()]

if atr_closed.empty:
    print('No closed ATR Candle trades with SL data.')
else:
    thresholds = range(20, 201, 10)
    results = []
    for t in thresholds:
        allowed = atr_closed[atr_closed['sl_pips'] <= t]
        results.append({
            'threshold': t,
            'trades_allowed': len(allowed),
            'trades_blocked': len(atr_closed) - len(allowed),
            'total_pips':     allowed['profit_pips'].sum(),
            'total_usd':      allowed['profit_usd'].sum(),
            'win_rate':       allowed['win'].mean() * 100 if len(allowed) > 0 else 0,
        })

    res = pd.DataFrame(results)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('ATR Candle — MaxSLPips Threshold Sensitivity', fontsize=13, fontweight='bold')

    axes[0].plot(res['threshold'], res['total_pips'], marker='o', color='steelblue')
    axes[0].axhline(0, color='black', linewidth=0.8)
    axes[0].set_title('Total P&L (pips)')
    axes[0].set_xlabel('MaxSLPips threshold')
    axes[0].set_ylabel('Pips')

    axes[1].plot(res['threshold'], res['win_rate'], marker='o', color='green')
    axes[1].axhline(50, color='orange', linestyle='--', linewidth=1)
    axes[1].set_title('Win Rate (%)')
    axes[1].set_xlabel('MaxSLPips threshold')
    axes[1].set_ylabel('%')
    axes[1].set_ylim(0, 105)

    axes[2].plot(res['threshold'], res['trades_allowed'], marker='o', color='steelblue', label='Allowed')
    axes[2].plot(res['threshold'], res['trades_blocked'], marker='x', color='red',      label='Blocked')
    axes[2].set_title('Trades Allowed vs Blocked')
    axes[2].set_xlabel('MaxSLPips threshold')
    axes[2].set_ylabel('Count')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

## Concurrent Order Analysis

Trades placed within 5 seconds across different magic numbers on the same symbol.  
These may have affected each other's fill prices — treat their slippage data with caution.

In [ ]:
conc_trades = trades[trades['concurrent']]

if conc_trades.empty:
    print('No concurrent orders detected.')
else:
    print(f'{len(conc_trades)} trades flagged as concurrent (within {CONCURRENT_SECONDS}s across magics).')
    print(f'\nConcurrent vs non-concurrent comparison (closed trades):')

    ct = closed_trades.copy()
    comp = ct.groupby('concurrent').agg(
        count    = ('win', 'count'),
        win_rate = ('win', lambda x: round(x.mean() * 100, 1)),
        avg_pips = ('profit_pips', 'mean'),
        avg_slip = ('slippage_pips', 'mean'),
    )
    comp.index = ['Non-concurrent', 'Concurrent']
    display(comp.style.format({
        'win_rate': '{:.1f}%',
        'avg_pips': '{:.1f}',
        'avg_slip': '{:.2f}',
    }, na_rep='-'))

    display(conc_trades[['log_time', 'magic', 'ticket', 'direction', 'method',
                          'sl_pips', 'slippage_pips', 'profit_pips']]
            .sort_values('log_time')
            .head(20)
            .style.set_caption('First 20 Concurrent Trades'))

## Efficiency Ratio at Entry vs Outcome

Tests whether higher ER at entry correlates with better trade outcomes — validates the ER filter thresholds.

In [ ]:
ct_er = closed_trades[closed_trades['er_value'].notna() & (closed_trades['er_value'] > 0)]

if ct_er.empty:
    print('No ER data in closed trades.')
else:
    ct_er = ct_er.copy()
    ct_er['er_bucket'] = pd.cut(ct_er['er_value'], bins=[0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.01],
                                 labels=['0-0.2','0.2-0.3','0.3-0.4','0.4-0.5',
                                         '0.5-0.6','0.6-0.7','0.7-0.8','0.8+'])

    er_stats = ct_er.groupby('er_bucket', observed=True).agg(
        count    = ('win', 'count'),
        win_rate = ('win', lambda x: x.mean() * 100),
        avg_pips = ('profit_pips', 'mean'),
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('Trade Outcome by ER Value at Entry', fontsize=13, fontweight='bold')

    axes[0].bar(er_stats.index, er_stats['win_rate'], color='steelblue', edgecolor='white')
    axes[0].axhline(50, color='orange', linestyle='--', linewidth=1)
    axes[0].set_title('Win Rate by ER Bucket')
    axes[0].set_xlabel('ER at Entry')
    axes[0].set_ylabel('Win Rate %')
    axes[0].set_ylim(0, 105)
    for i, (idx, row) in enumerate(er_stats.iterrows()):
        axes[0].text(i, row['win_rate'] + 1, str(int(row['count'])), ha='center', fontsize=9)

    axes[1].bar(er_stats.index, er_stats['avg_pips'],
                color=['#2ecc71' if v >= 0 else '#e74c3c' for v in er_stats['avg_pips']],
                edgecolor='white')
    axes[1].axhline(0, color='black', linewidth=0.8)
    axes[1].set_title('Avg P&L (pips) by ER Bucket')
    axes[1].set_xlabel('ER at Entry')
    axes[1].set_ylabel('Avg Pips')

    plt.tight_layout()
    plt.show()

## Raw Trade Table

In [ ]:
display_cols = ['magic', 'ticket', 'direction', 'method', 'log_time',
                'sl_pips', 'lots', 'spread_pips', 'atr_pips', 'er_value',
                'slippage_pips', 'hour', 'session',
                'close_reason', 'profit_pips', 'profit_usd', 'concurrent']

display_cols = [c for c in display_cols if c in trades.columns]

display(trades[display_cols]
    .sort_values('log_time', ascending=False)
    .head(50)
    .style
    .format({
        'sl_pips':      '{:.1f}',
        'lots':         '{:.2f}',
        'spread_pips':  '{:.1f}',
        'atr_pips':     '{:.1f}',
        'er_value':     '{:.3f}',
        'slippage_pips':'{:.1f}',
        'profit_pips':  '{:.1f}',
        'profit_usd':   '${:.2f}',
    }, na_rep='-')
    .applymap(lambda v: 'color: green' if isinstance(v, (int, float)) and v > 0
              else ('color: red' if isinstance(v, (int, float)) and v < 0 else ''),
              subset=['profit_pips', 'profit_usd'])
    .set_caption('Most Recent 50 Trades')
)